# Clustering replay debug

Inspect **stored DynamoDB state vs a TypeScript replay** of the production cleaning → clustering → categorisation pipeline.

## Workflow

1. On the **host** (repo root), with local DynamoDB running:

```bash
DYNAMODB_ENDPOINT=http://localhost:8000 DYNAMODB_TABLE_NAME=housef4-local-table \
  HOUSEF4_IMPORT_EMBEDDINGS=hash \
  pnpm --filter @housef4/backend run replay:clustering -- \
    --user-id local-dev \
    --json ml-training/generated/replay_clustering.json
```

Optional filters (clustering still uses the **full** corpus; filters only affect displayed rows):

```bash
# ... replay:clustering -- --user-id local-dev --merchant-substr SAINSBURYS --diffs-only --json ...
```

2. Refresh Postgres snapshot if needed: `docker compose exec ml-engine python scripts/extract_data_to_db.py`

3. Run the cells below.

**Note:** `replay.cluster_id` is reminted each run. Compare `physical_group_label`, `cleaned_merchant`, and category fields.

In [ ]:
import json
from pathlib import Path

import pandas as pd

REPLAY_JSON = Path('../generated/replay_clustering.json')

if not REPLAY_JSON.exists():
    raise FileNotFoundError(
        f'Missing {REPLAY_JSON.resolve()} — run replay:clustering on the host first (see markdown above).'
    )

payload = json.loads(REPLAY_JSON.read_text())
meta = payload['meta']
print('Replay meta:', json.dumps(meta, indent=2))

rows = payload['rows']
df = pd.json_normalize(rows, sep='.')
print(f'Loaded {len(df)} replay rows')

In [ ]:
DIFF_COLS = [
    'differs.cleaned_merchant',
    'differs.category',
    'differs.status',
    'differs.suggested_category',
    'differs.match_type',
    'differs.grouping',
]

df['any_diff'] = df[DIFF_COLS].any(axis=1)
print(f"Rows with any diff: {df['any_diff'].sum()} / {len(df)}")

view_cols = [
    'id',
    'raw_merchant',
    'stored.cleaned_merchant',
    'replay.cleaned_merchant',
    'stored.category',
    'replay.category',
    'stored.status',
    'replay.status',
    'stored.suggested_category',
    'replay.suggested_category',
    'replay.match_type',
    'replay.physical_group_label',
    'replay.physical_group_size',
    'stored.cluster_id',
    'any_diff',
]

display(df.loc[df['any_diff'], view_cols].head(30))

In [ ]:
# Optional: join Postgres snapshot (includes stored cluster/category fields from DynamoDB)
import os

import psycopg2

PG_COLS = """
    transaction_id, date, amount, description, cleaned_merchant, cluster_id,
    category, status, suggested_category, category_confidence, match_type,
    pairing_id, transaction_file_id
"""

conn = psycopg2.connect(
    host=os.environ.get('DB_HOST', 'db'),
    port=os.environ.get('DB_PORT', '5432'),
    user=os.environ.get('DB_USER', 'ml_user'),
    password=os.environ.get('DB_PASSWORD', 'ml_password'),
    database=os.environ.get('DB_NAME', 'recurring_charges_ml'),
)

df_pg = pd.read_sql(f'SELECT {PG_COLS} FROM transactions', conn)
merged = df.merge(df_pg, left_on='id', right_on='transaction_id', how='left', suffixes=('', '_pg'))
display(merged.loc[merged['any_diff'], view_cols + ['date', 'amount', 'cluster_id']].head(20))

In [ ]:
# Drill into one transaction
TXN_ID = None  # e.g. 'txn-abc123'

if TXN_ID:
    hit = df[df['id'] == TXN_ID]
    if hit.empty:
        print('Not in replay output — widen filters when generating JSON')
    else:
        display(hit.T)
else:
    print('Set TXN_ID to inspect a single row.')